# ViettelRace — Fine-tune joint NER + assertion model

Chạy notebook này trên **Kaggle** (Settings → Accelerator → GPU T4 x2 hoặc P100) hoặc **Colab**
(Runtime → Change runtime type → GPU).

**Input cần có** (upload làm Kaggle Dataset, hoặc lên Colab qua `files.upload()` / Google Drive):
- `train.jsonl`, `holdout.jsonl` — sinh bởi `scripts/prepare_ner_dataset.py` ở repo, đặt trong
  thư mục `/kaggle/input/viettelrace-ner-dataset/` (Kaggle) hoặc `data/ner_dataset/` (Colab).

**Output**: một thư mục model đã fine-tune, nén lại để tải về và đặt vào `models/ner_model/`
trong repo, dùng bởi `scripts/run_pipeline.py`.

Model nền: `xlm-roberta-base` — chọn vì có fast tokenizer hỗ trợ `offset_mapping` trực tiếp trên
văn bản tiếng Việt thô (không cần qua bước word-segmentation như PhoBERT), nên việc map lại
`position` [start,end] ở output đơn giản và chính xác hơn.

Kiến trúc: 1 encoder dùng chung, 2 đầu ra:
1. **BIO tagging** (11 nhãn: O + B-/I- × 5 loại) → xác định `text`, `type`, `position`.
2. **Assertion multi-label** (isNegated / isHistorical / isFamily) trên từng token, chỉ tính
   loss trên token thuộc thực thể loại CHẨN_ĐOÁN/TRIỆU_CHỨNG/THUỐC.

`candidates` (mã ICD/RxNorm) **không** sinh ra từ model này — đó là bài toán entity-linking,
xử lý riêng ở `scripts/build_terminology_index.py` + `scripts/run_pipeline.py`.


In [ ]:
# transformers/torch/numpy are already preinstalled on Kaggle's and Colab's
# standard images -- this notebook doesn't use datasets/seqeval/accelerate,
# so no install step is needed (and Kaggle kernels without internet access
# enabled, e.g. unverified accounts, would fail here otherwise).
import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__)


In [ ]:
import json
import os
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

SEED = 13
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Cheap first filter only -- a trivial add is not a reliable GPU health check.
# Some Kaggle GPU allocations (e.g. older Tesla P100, compute capability
# sm_60) are no longer supported by the newest preinstalled torch wheel for
# the specific ops a transformer forward pass needs, even though a bare
# tensor add works fine. That combination previously passed this probe and
# then crashed ~40s into epoch 1 with "no kernel image is available for
# execution on the device" (see kaggle_upload/kernel/run_output/*.log). The
# real check -- a full forward+backward through the actual model -- runs in
# the next cell, right after the model is built and before the training
# loop starts, so a bad GPU is caught in seconds instead of after a partial
# epoch.
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("candidate device (unverified):", DEVICE)

MODEL_NAME = "xlm-roberta-base"
# 320 was silently truncating training AND inference input: 46/100 corpus docs
# exceed 320 tokens (median doc is already 316), which dropped 640/2223
# (28.8%) of ground-truth entities entirely -- the model never saw them and
# could never predict them, regardless of how well it otherwise trained.
# xlm-roberta-base's position embeddings cap out at 512 -- the max reachable
# without a sliding-window/chunking rewrite -- which cuts entity loss to
# 237/2223 (10.7%). Confirmed by tokenizing every input/*.txt and counting
# output/*.json entities starting past each cutoff.
MAX_LENGTH = 512
# 21/100 corpus docs are still longer than MAX_LENGTH even at 512. Rather than
# leave that residual 10.7% entity loss in place, both training (see
# window_records below) and inference (scripts/run_pipeline.py) now split
# long documents into overlapping windows instead of truncating. The overlap
# only needs to be longer than the longest entity mention in the corpus (a
# few dozen characters) for every entity to land fully inside at least one
# window regardless of where in the document it sits.
STRIDE = 64
ASSERTION_LOSS_WEIGHT = 0.5
BATCH_SIZE = 8
EPOCHS = 20
LR = 2e-5
MIN_EPOCHS = 8
EARLY_STOP_PATIENCE = 4
MIN_HOLDOUT_DELTA = 0.01
ASSERTION_THRESHOLD_CANDIDATES = [0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]
DEFAULT_ASSERTION_THRESHOLD = 0.45
FINAL_TRAIN_EPOCHS = 16


In [ ]:
# Locate the dataset. Kaggle's actual mount path for API-pushed datasets can
# be /kaggle/input/datasets/&lt;username&gt;/&lt;slug&gt; rather than the usual
# /kaggle/input/&lt;slug&gt; used for datasets added via the "Add Data" UI -- glob
# for it recursively instead of hardcoding one path.
import glob

matches = glob.glob("/kaggle/input/**/train.jsonl", recursive=True) if Path("/kaggle/input").exists() else []
CANDIDATE_DIRS = [Path(m).parent for m in matches] + [
    Path("data/ner_dataset"),
    Path("/content/data/ner_dataset"),
]
DATA_DIR = next((d for d in CANDIDATE_DIRS if (d / "train.jsonl").exists()), None)
if DATA_DIR is None:
    if Path("/kaggle/input").exists():
        print("contents of /kaggle/input:", os.listdir("/kaggle/input"))
    raise FileNotFoundError(
        "train.jsonl not found. Upload data/ner_dataset/{train,holdout}.jsonl "
        "(generated locally by scripts/prepare_ner_dataset.py) as a Kaggle Dataset "
        "or into the Colab file browser, then update CANDIDATE_DIRS above."
    )
print("Using dataset dir:", DATA_DIR)


def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]


train_records = load_jsonl(DATA_DIR / "train.jsonl")
holdout_records = load_jsonl(DATA_DIR / "holdout.jsonl")
print(f"train: {len(train_records)} files, holdout: {len(holdout_records)} files")


def base_record_id(value):
    return str(value).split("-", 1)[0]


train_base_ids = {base_record_id(r["id"]) for r in train_records}
holdout_base_ids = {base_record_id(r["id"]) for r in holdout_records}
TRAIN_HOLDOUT_OVERLAP = bool(train_base_ids & holdout_base_ids)
if TRAIN_HOLDOUT_OVERLAP:
    print(
        "final train-all mode: train.jsonl overlaps holdout.jsonl; "
        "holdout is reported only and checkpoint selection uses fixed epoch."
    )


In [ ]:
ENTITY_TYPES = ["CHẨN_ĐOÁN", "TRIỆU_CHỨNG", "THUỐC", "TÊN_XÉT_NGHIỆM", "KẾT_QUẢ_XÉT_NGHIỆM"]
ASSERTION_TYPES = {"CHẨN_ĐOÁN", "TRIỆU_CHỨNG", "THUỐC"}
ASSERTION_LABELS = ["isNegated", "isHistorical", "isFamily"]

BIO_LABELS = ["O"] + [f"{prefix}-{t}" for t in ENTITY_TYPES for prefix in ("B", "I")]
label2id = {l: i for i, l in enumerate(BIO_LABELS)}
id2label = {i: l for l, i in label2id.items()}
print(f"{len(BIO_LABELS)} BIO labels:", BIO_LABELS)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)


In [ ]:
def window_records(example, tokenizer, max_length=MAX_LENGTH, stride=STRIDE):
    """Split one document into overlapping windows so no training example
    exceeds the model's position-embedding limit. MAX_LENGTH=512 alone still
    leaves 21/100 corpus documents too long -- without this, every entity
    past the cutoff is silently dropped from training, the same failure mode
    that MAX_LENGTH=320 had (see CLAUDE.md / worklog.md). An entity is kept
    in a window only when its full span falls inside it; one that straddles
    a boundary is dropped from that window rather than truncated into a
    corrupt BIO label -- it appears whole in a neighboring window instead,
    since windows overlap by `stride` tokens, comfortably larger than any
    entity mention in this corpus.
    """
    text = example["text"]
    offsets = tokenizer(text, return_offsets_mapping=True, truncation=False, add_special_tokens=False)[
        "offset_mapping"
    ]
    n = len(offsets)
    content_budget = max_length - 2
    if n <= content_budget:
        return [example]

    step = content_budget - stride
    windows = []
    start = 0
    while start < n:
        end = min(start + content_budget, n)
        char_start, char_end = offsets[start][0], offsets[end - 1][1]
        window_entities = []
        for ent in example["entities"]:
            if ent["start"] >= char_start and ent["end"] <= char_end:
                e2 = dict(ent)
                e2["start"] -= char_start
                e2["end"] -= char_start
                window_entities.append(e2)
        windows.append(
            {"id": f"{example['id']}-w{len(windows)}", "text": text[char_start:char_end], "entities": window_entities}
        )
        if end == n:
            break
        start += step
    return windows


def encode_example(example, tokenizer, max_length=MAX_LENGTH):
    text = example["text"]
    enc = tokenizer(
        text,
        return_offsets_mapping=True,
        truncation=True,
        max_length=max_length,
    )
    offsets = enc.pop("offset_mapping")
    seq_len = len(enc["input_ids"])

    bio_ids = [label2id["O"]] * seq_len
    assertion_vec = [[0, 0, 0] for _ in range(seq_len)]
    assertion_mask = [0] * seq_len
    ner_loss_mask = [0 if (s == 0 and e == 0) else 1 for s, e in offsets]

    entities = sorted(example["entities"], key=lambda e: e["start"])
    for ent in entities:
        s, e, typ = ent["start"], ent["end"], ent["type"]
        first = True
        for idx, (tok_s, tok_e) in enumerate(offsets):
            if tok_s == tok_e == 0:
                continue
            if tok_e <= s or tok_s >= e:
                continue
            label = f"{'B' if first else 'I'}-{typ}"
            bio_ids[idx] = label2id[label]
            first = False
            if typ in ASSERTION_TYPES:
                assertion_mask[idx] = 1
                for a in ent.get("assertions", []):
                    if a in ASSERTION_LABELS:
                        assertion_vec[idx][ASSERTION_LABELS.index(a)] = 1

    ner_labels = [bio_ids[i] if ner_loss_mask[i] else -100 for i in range(seq_len)]

    enc["ner_labels"] = ner_labels
    enc["assertion_labels"] = assertion_vec
    enc["assertion_mask"] = assertion_mask
    enc["offsets"] = offsets
    return enc


class NerDataset(Dataset):
    def __init__(self, records, tokenizer, max_length=MAX_LENGTH, window=False):
        if window:
            expanded = []
            for r in records:
                expanded.extend(window_records(r, tokenizer, max_length))
            n_extra = len(expanded) - len(records)
            if n_extra:
                print(f"windowed {len(records)} docs -> {len(expanded)} training examples (+{n_extra} from long docs)")
            records = expanded
        self.examples = [encode_example(r, tokenizer, max_length) for r in records]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def collate_fn(batch, pad_id):
    max_len = max(len(ex["input_ids"]) for ex in batch)

    def pad(seq, value):
        return seq + [value] * (max_len - len(seq))

    input_ids = torch.tensor([pad(ex["input_ids"], pad_id) for ex in batch])
    attention_mask = torch.tensor([pad(ex["attention_mask"], 0) for ex in batch])
    ner_labels = torch.tensor([pad(ex["ner_labels"], -100) for ex in batch])
    assertion_mask = torch.tensor([pad(ex["assertion_mask"], 0) for ex in batch]).float()
    assertion_labels = torch.tensor(
        [pad(ex["assertion_labels"], [0, 0, 0]) for ex in batch]
    ).float()
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "ner_labels": ner_labels,
        "assertion_mask": assertion_mask,
        "assertion_labels": assertion_labels,
    }


# window=True only for train -- holdout stays doc-level since predict_records
# (below) does its own sliding-window inference + merge per whole document,
# matching how scripts/run_pipeline.py scores a real submission.
train_ds = NerDataset(train_records, tokenizer, window=True)
holdout_ds = NerDataset(holdout_records, tokenizer, window=False)

pad_id = tokenizer.pad_token_id
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=lambda b: collate_fn(b, pad_id),
)
holdout_loader = DataLoader(
    holdout_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=lambda b: collate_fn(b, pad_id),
)


In [ ]:
class JointNerAssertionModel(nn.Module):
    def __init__(self, model_name, num_bio_labels, num_assertion_labels, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.ner_head = nn.Linear(hidden, num_bio_labels)
        self.assertion_head = nn.Linear(hidden, num_assertion_labels)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden = self.dropout(out.last_hidden_state)
        return self.ner_head(hidden), self.assertion_head(hidden)


model = JointNerAssertionModel(MODEL_NAME, len(BIO_LABELS), len(ASSERTION_LABELS))


def device_actually_works(model: nn.Module, device: torch.device) -> bool:
    """Run one real forward+backward on a dummy batch on `device`. A device
    that fails here (rather than on the cheap probe in the previous cell) is
    the exact failure mode that crashed the 2026-07 Kaggle run mid-epoch --
    catch it before touching the real training loop.
    """
    try:
        model.to(device)
        dummy_ids = torch.ones((2, 8), dtype=torch.long, device=device)
        dummy_mask = torch.ones((2, 8), dtype=torch.long, device=device)
        ner_logits, assertion_logits = model(dummy_ids, dummy_mask)
        (ner_logits.sum() + assertion_logits.sum()).backward()
        model.zero_grad(set_to_none=True)
        return True
    except RuntimeError as exc:
        print(f"Device {device} failed a real forward/backward probe ({exc}); falling back to CPU.")
        return False


if DEVICE.type == "cuda" and not device_actually_works(model, DEVICE):
    DEVICE = torch.device("cpu")

model = model.to(DEVICE)
print("training device:", DEVICE)

ner_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
assertion_loss_fn = nn.BCEWithLogitsLoss(reduction="none")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)


In [ ]:
# Holdout scoring helpers used both during training and after checkpoint selection.
# The metric mirrors the model-owned part of the leaderboard score:
#   30 * (1 - WER) + 30 * J_assertion
# Candidate linking is handled by the terminology step, so it is intentionally
# not included in checkpoint selection.

def decode_entities(text, offsets, bio_ids, assertion_probs, threshold=0.5):
    entities = []
    i = 0
    n = len(offsets)
    while i < n:
        tok_s, tok_e = offsets[i]
        if tok_s == tok_e == 0:
            i += 1
            continue
        label = id2label[bio_ids[i]]
        if label == "O":
            i += 1
            continue
        _, typ = label.split("-", 1)
        start = tok_s
        end = tok_e
        probs_acc = [assertion_probs[i]]
        j = i + 1
        while j < n:
            ntok_s, ntok_e = offsets[j]
            if ntok_s == ntok_e == 0:
                break
            nlabel = id2label[bio_ids[j]]
            if nlabel != f"I-{typ}":
                break
            end = ntok_e
            probs_acc.append(assertion_probs[j])
            j += 1
        mean_probs = np.mean(probs_acc, axis=0)
        assertions = [ASSERTION_LABELS[k] for k, p in enumerate(mean_probs) if p >= threshold]
        ent = {"start": start, "end": end, "type": typ, "text": text[start:end]}
        if typ in ASSERTION_TYPES:
            ent["assertions"] = assertions
        entities.append(ent)
        i = j
    return entities


def sliding_windows(n_tokens, max_len, stride):
    if n_tokens <= max_len:
        return [(0, n_tokens)]
    step = max_len - stride
    windows = []
    start = 0
    while start < n_tokens:
        end = min(start + max_len, n_tokens)
        windows.append((start, end))
        if end == n_tokens:
            break
        start += step
    return windows


def dedupe_and_resolve(entities):
    best = {}
    for e in entities:
        key = (e["start"], e["end"], e["type"])
        if key not in best:
            best[key] = dict(e)
        elif e["type"] in ASSERTION_TYPES:
            best[key]["assertions"] = sorted(set(best[key].get("assertions", [])) | set(e.get("assertions", [])))
    ordered = sorted(best.values(), key=lambda e: (e.get("_edge", False), -(e["end"] - e["start"])))
    kept = []
    for e in ordered:
        if any(max(e["start"], k["start"]) < min(e["end"], k["end"]) for k in kept):
            continue
        kept.append(e)
    kept.sort(key=lambda e: (e["start"], e["end"]))
    for e in kept:
        e.pop("_edge", None)
    return kept


@torch.no_grad()
def predict_records(records, tokenizer, threshold=0.5, max_length=MAX_LENGTH, stride=STRIDE):
    model.eval()
    predictions = []
    content_budget = max_length - 2
    for rec in records:
        text = rec["text"]
        full_offsets = tokenizer(text, return_offsets_mapping=True, truncation=False, add_special_tokens=False)[
            "offset_mapping"
        ]
        token_windows = sliding_windows(len(full_offsets), content_budget, stride)

        all_entities = []
        for w_idx, (w_start, w_end) in enumerate(token_windows):
            char_start, char_end = full_offsets[w_start][0], full_offsets[w_end - 1][1]
            is_first, is_last = w_idx == 0, w_idx == len(token_windows) - 1
            window_text = text[char_start:char_end]

            enc = tokenizer(window_text, return_offsets_mapping=True, truncation=True, max_length=max_length)
            raw_offsets = enc.pop("offset_mapping")
            offsets = [(0, 0) if s == e == 0 else (s + char_start, e + char_start) for s, e in raw_offsets]
            input_ids = torch.tensor([enc["input_ids"]]).to(DEVICE)
            attention_mask = torch.tensor([enc["attention_mask"]]).to(DEVICE)
            ner_logits, assertion_logits = model(input_ids, attention_mask)
            bio_ids = ner_logits.argmax(dim=-1)[0].cpu().numpy().tolist()
            assertion_probs = torch.sigmoid(assertion_logits)[0].cpu().numpy()

            window_entities = decode_entities(text, offsets, bio_ids, assertion_probs, threshold=threshold)
            for e in window_entities:
                e["_edge"] = (not is_first and e["start"] <= char_start + 2) or (
                    not is_last and e["end"] >= char_end - 2
                )
            all_entities.extend(window_entities)

        entities = dedupe_and_resolve(all_entities)
        predictions.append({"id": rec["id"], "entities": entities})
    return predictions


def edit_distance(a, b):
    if not a:
        return len(b)
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ai in enumerate(a, 1):
        cur = [i] + [0] * len(b)
        for j, bj in enumerate(b, 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (0 if ai == bj else 1))
        prev = cur
    return prev[-1]


def wer(ref, hyp):
    if not ref:
        return 0.0 if not hyp else 1.0
    return edit_distance(ref, hyp) / len(ref)


def jaccard(a, b):
    if not a and not b:
        return 1.0
    if not a and b:
        return 0.0
    return len(a & b) / len(a | b)


def words_for_metric(entities):
    ordered = sorted(entities, key=lambda e: (e["start"], e["end"], e["type"], e["text"]))
    return " ".join(e["text"] for e in ordered).split()


def assertion_items(entities):
    items = set()
    for e in entities:
        if e["type"] not in ASSERTION_TYPES:
            continue
        base = (e["text"], e["type"])
        a = e.get("assertions") or []
        if not a:
            items.add((*base, "__EMPTY__"))
        else:
            items.update((*base, x) for x in a)
    return items


def score_holdout(records, tokenizer, thresholds=(0.5,)):
    gt_by_id = {r["id"]: r["entities"] for r in records}
    best = None
    for threshold in thresholds:
        pred_by_id = {p["id"]: p["entities"] for p in predict_records(records, tokenizer, threshold=threshold)}
        wer_scores, assertion_scores = [], []
        for fid, gt_entities in gt_by_id.items():
            pred_entities = pred_by_id[fid]
            wer_scores.append(wer(words_for_metric(gt_entities), words_for_metric(pred_entities)))
            assertion_scores.append(jaccard(assertion_items(gt_entities), assertion_items(pred_entities)))
        mean_wer = float(np.mean(wer_scores))
        mean_assertion = float(np.mean(assertion_scores))
        model_points = 30.0 * (1.0 - mean_wer) + 30.0 * mean_assertion
        metrics = {
            "threshold": float(threshold),
            "wer": mean_wer,
            "j_assertion": mean_assertion,
            "model_points": float(model_points),
        }
        if best is None or metrics["model_points"] > best["model_points"]:
            best = metrics
    return best


In [ ]:
def compute_loss(batch):
    input_ids = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)
    ner_labels = batch["ner_labels"].to(DEVICE)
    assertion_mask = batch["assertion_mask"].to(DEVICE)
    assertion_labels = batch["assertion_labels"].to(DEVICE)

    ner_logits, assertion_logits = model(input_ids, attention_mask)

    ner_loss = ner_loss_fn(ner_logits.view(-1, len(BIO_LABELS)), ner_labels.view(-1))

    per_token_assertion_loss = assertion_loss_fn(assertion_logits, assertion_labels).mean(dim=-1)
    denom = assertion_mask.sum().clamp(min=1.0)
    assertion_loss = (per_token_assertion_loss * assertion_mask).sum() / denom

    return ner_loss + ASSERTION_LOSS_WEIGHT * assertion_loss, ner_loss.item(), assertion_loss.item()


def snapshot_state_dict(module):
    return {k: v.detach().cpu().clone() for k, v in module.state_dict().items()}


best_state_dict = None
best_epoch = 0
best_holdout = {"model_points": -1.0, "wer": 1.0, "j_assertion": 0.0, "threshold": DEFAULT_ASSERTION_THRESHOLD}
stale_epochs = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total, total_ner, total_assert = 0.0, 0.0, 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        loss, ner_l, assert_l = compute_loss(batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total += loss.item()
        total_ner += ner_l
        total_assert += assert_l
    n = len(train_loader)

    holdout = score_holdout(holdout_records, tokenizer, thresholds=(0.5,))

    if TRAIN_HOLDOUT_OVERLAP:
        best_state_dict = snapshot_state_dict(model)
        best_epoch = epoch
        best_holdout = dict(holdout)
        improved = epoch == 1
        stale_epochs = 0
    else:
        improved = holdout["model_points"] > best_holdout["model_points"] + MIN_HOLDOUT_DELTA
        if improved:
            best_holdout = dict(holdout)
            best_epoch = epoch
            best_state_dict = snapshot_state_dict(model)
            stale_epochs = 0
        else:
            stale_epochs += 1

    print(
        f"epoch {epoch:2d}: loss={total/n:.4f} ner={total_ner/n:.4f} assertion={total_assert/n:.4f} | "
        f"holdout WER={holdout['wer']:.4f} J_assertion={holdout['j_assertion']:.4f} "
        f"model_points={holdout['model_points']:.2f}/60 | best_epoch={best_epoch} "
        f"best_points={best_holdout['model_points']:.2f}/60 stale={stale_epochs}"
    )

    if TRAIN_HOLDOUT_OVERLAP and epoch >= FINAL_TRAIN_EPOCHS:
        print(f"final train-all mode: stopping at fixed epoch {FINAL_TRAIN_EPOCHS}")
        break
    if not TRAIN_HOLDOUT_OVERLAP and epoch >= MIN_EPOCHS and stale_epochs >= EARLY_STOP_PATIENCE:
        print(f"early stopping at epoch {epoch}; no holdout improvement for {stale_epochs} epochs")
        break

if best_state_dict is not None:
    model.load_state_dict(best_state_dict)

BEST_EPOCH = best_epoch or epoch
if TRAIN_HOLDOUT_OVERLAP:
    ASSERTION_THRESHOLD = DEFAULT_ASSERTION_THRESHOLD
    BEST_HOLDOUT = score_holdout(holdout_records, tokenizer, thresholds=(ASSERTION_THRESHOLD,))
else:
    BEST_HOLDOUT = score_holdout(holdout_records, tokenizer, thresholds=ASSERTION_THRESHOLD_CANDIDATES)
    ASSERTION_THRESHOLD = BEST_HOLDOUT["threshold"]
print(
    f"selected checkpoint epoch={BEST_EPOCH}, assertion_threshold={ASSERTION_THRESHOLD:.2f}, "
    f"holdout WER={BEST_HOLDOUT['wer']:.4f}, J_assertion={BEST_HOLDOUT['j_assertion']:.4f}, "
    f"model_points={BEST_HOLDOUT['model_points']:.2f}/60"
)


In [ ]:
# Report the selected checkpoint. Training above already loads the best
# holdout checkpoint back into `model` and calibrates ASSERTION_THRESHOLD.
print(f"holdout mean WER (span+type only, no candidates): {BEST_HOLDOUT['wer']:.4f}")
print(f"holdout mean J_assertion: {BEST_HOLDOUT['j_assertion']:.4f}")
print(f"selected assertion threshold: {ASSERTION_THRESHOLD:.2f}")
print("(candidates score is not estimated here -- that comes from the terminology")
print(" matcher in scripts/build_terminology_index.py, evaluated after linking.)")


In [ ]:
# Export the selected checkpoint + label maps for use by scripts/run_pipeline.py.
EXPORT_DIR = Path("ner_model_export")
EXPORT_DIR.mkdir(exist_ok=True)

# The training loop has already restored the best holdout checkpoint into model.
torch.save(model.state_dict(), EXPORT_DIR / "model.pt")
tokenizer.save_pretrained(EXPORT_DIR)
model.encoder.config.to_json_file(EXPORT_DIR / "hf_config.json")
(EXPORT_DIR / "config.json").write_text(json.dumps({
    "base_model": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "stride": STRIDE,
    "bio_labels": BIO_LABELS,
    "assertion_labels": ASSERTION_LABELS,
    "entity_types": ENTITY_TYPES,
    "assertion_threshold": float(globals().get("ASSERTION_THRESHOLD", 0.5)),
    "best_epoch": int(globals().get("BEST_EPOCH", EPOCHS)),
    "best_holdout": globals().get("BEST_HOLDOUT", {}),
    "train_holdout_overlap": bool(globals().get("TRAIN_HOLDOUT_OVERLAP", False)),
}, ensure_ascii=False, indent=2), encoding="utf-8")

import shutil
shutil.make_archive("ner_model_export", "zip", EXPORT_DIR)
print("Wrote ner_model_export.zip -- download it and unzip into models/ner_model/ in the repo.")


## Sau khi tải model về

1. Giải nén `ner_model_export.zip` vào `models/ner_model/` trong repo.
2. Chạy `python scripts/run_pipeline.py` (đọc `input/`, chạy model + terminology matcher, ghi
   `output_model/`).
3. Validate bằng `python scripts/check_submission.py --pred output_model --input input`.
4. So sánh `output_model/` với `output/` (bản thủ công đã chấm 41.59) trên vài file để đánh giá
   trực quan trước khi quyết định thay thế bản nộp.

Số liệu `holdout` ở notebook này chỉ là ước lượng nội bộ trên 15 file bị giữ lại khỏi tập train —
không thay thế được điểm leaderboard thật, nhưng giúp phát hiện overfit/underfit trước khi nộp.
